# Amenity Index Creation

Here I'll be combining data from the `city_facilities_data_clean_analysis.ipynb` and `sf_acs_walkscore.ipynb` notebooks to construct an index for each amenity category that comprises of gross square feet of amenities, population totals, gini index, median income, poverty count, % white/non hispanic, transit commuters, and more.

In [63]:
# import necessary dependencies 
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
from census import Census
from us import states
import os
from pathlib import Path
from IPython.display import display
import seaborn as sns
import re
import zipfile
from shapely.geometry import Point
import folium
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib as mpl

# set general seaborn style
sns.set_theme(style="whitegrid", font_scale=1.1)

In [64]:
# uploading processed data from previous notebooks

# use robust path handling to locate the data file
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'requirements.txt').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()

amenities_gdf = gpd.read_file(repo_root / 'data' / 'processed' / 'city_facilities_cleaned.geojson', engine="pyogrio")
walkscore_census_df = gpd.read_file(repo_root / 'data' / 'processed' / 'final_sf_acs_years_panel.csv', engine="pyogrio")

#### Combine Data 

First, I'll take another glance at the data to be able to combine them by tract, so I can actually create an index.

I'll start with looking at the `amenities_gdf` file.

In [65]:
# see the first few rows of amenities_gdf
amenities_gdf.head()

,:id,:version,facility_id,common_name,address,city,zip_code,block_lot,owned_leased,dept_id,...,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,amenity_category,geometry
0,row-s7n8_dawh~di6g,rv-cqxy.cfcc_9h38,3199,SE CENTRIFUGAL BLDG - 840,1800 Oakdale Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39422 37.73832)
1,row-bhtc-9uc2-x9yi,rv-65e3-cxdx.w4qh,1348,Wholesale Produce Market - Public Dock 2 Middle,2002 Jerrold Ave,San Francisco,94124,5281020,Own,63,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39736 37.74330)
2,row-y6zf-u23p.6mxm,rv-4rqu.rsa9~zrey,3181,SE SED BLDG #3 042,1700 Jerrold Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39112 37.74330)
3,row-jx69-xpuh-gzwy,rv-ujtr-73h5-tc8h,3187,SE D. SLUDGE THK. TANK - 750,1700 Jerrold Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.38981 37.74273)
4,row-gsda_jc6m~mcgx,rv-6d7q~5xph-y6a9,1808,Urban Forestry Crew Shack,2323 Cesar Chavez St,San Francisco,94124,4341001,Own,77,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.40040 37.74905)


In [66]:
# see the columns of amenities_gdf
amenities_gdf.columns

Index([':id', ':version', 'facility_id', 'common_name', 'address', 'city',
       'zip_code', 'block_lot', 'owned_leased', 'dept_id', 'department_name',
       'gross_sq_ft', 'longitude', 'latitude', 'supervisor_district',
       'city_tenants', 'land_id', ':@computed_region_ajp5_b2md',
       ':@computed_region_f58d_8dbm', ':@computed_region_rxqg_mtj9',
       ':@computed_region_jx4q_fizf', ':@computed_region_yftq_j783',
       ':@computed_region_bh8s_q3mv', ':@computed_region_jwn9_ihcz',
       ':@computed_region_6qbp_sg9q', ':@computed_region_qgnn_b9vv',
       ':@computed_region_26cr_cadq', 'index_right', 'STATEFP', 'COUNTYFP',
       'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT',
       'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'amenity_category',
       'geometry'],
      dtype='object')

In [67]:
# see the shape of amenities_gdf
amenities_gdf.shape

(1316, 43)

In [68]:
# see the number of unique tracts in amenities_gdf
amenities_gdf['GEOID'].nunique()

209

Now I'll move onto taking a look at the `walkscore_census_df` file.

In [69]:
# see the first few rows of walkscore_census_df
walkscore_census_df.head()

,NAME,population_total,white/non_hispanic,gini_index,median_income,poverty_denominator,poverty_count,transit_commuters,state,county,tract,GEOID,year,%_white_nonhisp,poverty_rate
0,"Census Tract 260.04, San Francisco County, Cal...",5015.0,528.0,0.4582,65862.0,5015.0,456.0,1099.0,06,075,026004,06075026004,2015,0.10528414755732801,0.0909272183449651
1,"Census Tract 301.01, San Francisco County, Cal...",4895.0,2915.0,0.4959,79071.0,4735.0,583.0,1326.0,06,075,030101,06075030101,2015,0.5955056179775281,0.12312565997888067
2,"Census Tract 330, San Francisco County, Califo...",8227.0,2891.0,0.4607,82527.0,8162.0,1088.0,1093.0,06,075,033000,06075033000,2015,0.3514039139418986,0.1333006616025484
3,"Census Tract 254.03, San Francisco County, Cal...",5154.0,1168.0,0.4495,73159.0,5128.0,611.0,1018.0,06,075,025403,06075025403,2015,0.22662010089251067,0.11914976599063963
4,"Census Tract 264.01, San Francisco County, Cal...",3937.0,137.0,0.4317,46150.0,3937.0,501.0,624.0,06,075,026401,06075026401,2015,0.034798069596139194,0.12725425450850902


In [70]:
# see columns of walkscore_census_df
walkscore_census_df.columns

Index(['NAME', 'population_total', 'white/non_hispanic', 'gini_index',
       'median_income', 'poverty_denominator', 'poverty_count',
       'transit_commuters', 'state', 'county', 'tract', 'GEOID', 'year',
       '%_white_nonhisp', 'poverty_rate'],
      dtype='object')

In [71]:
# see the shape of walkscore_census_df
walkscore_census_df.shape

(1088, 15)

In [72]:
# see the years available in walkscore_census_df
walkscore_census_df['year'].value_counts()

year
2020    236
2022    236
2023    236
2015    190
2018    190
Name: count, dtype: int64

We'll use the data from the most recent year (2020) to merge with amenity related features.

In [73]:
# get walkscore census data from 2020
walkscore_2020_df = walkscore_census_df[walkscore_census_df['year'] == "2020"]

In [74]:
# merge amenities_gdf with walkscore_2020_df on GEOID
merged_df = amenities_gdf[["GEOID", "gross_sq_ft", "amenity_category"]].merge(walkscore_2020_df, on="GEOID")

Now with a merged DataFrame, I'll move onto standardizing (normalizing) the variables below and calculating an index as well as adjusting them with the population_totals.
- 30% gross_sq_feet --> larger is better
- 15% gini_index --> larger is not better, so multiply by -1 too
- 15% white/non_hispanic --> larger is better
- 5% poverty_count --> larger is not better, so multiply by -1
- 15% transit_commuters --> larger is better
- 20% population_totals --> neutral

In [87]:
# convert gini_index, white/non_hispanic, poverty_count, population_totals, transit_commuters to numeric data types in merged_df
merged_df['gini_index'] = pd.to_numeric(merged_df['gini_index'], errors='coerce')            
merged_df['white/non_hispanic'] = pd.to_numeric(merged_df['white/non_hispanic'], errors='coerce')
merged_df['poverty_count'] = pd.to_numeric(merged_df['poverty_count'], errors='coerce')
merged_df['population_total'] = pd.to_numeric(merged_df['population_total'], errors='coerce')
merged_df['transit_commuters'] = pd.to_numeric(merged_df['transit_commuters'], errors='coerce')

With the data types being converted to numeric values, I'll work on standardizing the values, constructing the index, and evaluating key metrics of the index.

In [100]:
# standardize variables within each amenity_category (preserves tract-level rows,
# but uses category-specific mean/std for each variable)

# ensure amenity_category exists
merged_df["amenity_category"] = merged_df["amenity_category"].fillna("Unknown")

# copy to hold standardized columns
merged_df_std = merged_df.copy()

# columns to standardize and whether to invert (True => multiply by -1 before stats)
vars_to_std = {
    "gross_sq_ft": False,               # larger is better
    "gini_index": True,                 # larger is worse -> invert
    "white/non_hispanic": False,        # larger is better
    "poverty_count": True,              # larger is worse -> invert
    "transit_commuters": False,         # larger is better
    "population_total": False           # neutral but still standardized
}

# create placeholder std columns
for col in vars_to_std:
    out_col = f"{col}_std"
    merged_df_std[out_col] = np.nan

# group by category and standardize within-group
for cat, grp in merged_df.groupby("amenity_category", sort=False):
    # indices of this group
    idx = grp.index
    # for each variable, compute mean/std on transformed values and assign z-scores
    for col, invert in vars_to_std.items():
        # prepare series: coerce to numeric to be safe
        s = pd.to_numeric(grp[col], errors="coerce").astype(float)
        if invert:
            s_trans = -1.0 * s
        else:
            s_trans = s
        mean = s_trans.mean(skipna=True)
        std = s_trans.std(skipna=True)
        # avoid division by zero / NaN: if std is 0 or NaN, set to 1 so z-score -> 0 for constant groups
        if not np.isfinite(std) or std == 0:
            std = 1.0
        z = (s_trans - mean) / std
        merged_df_std.loc[idx, f"{col}_std"] = z.values

# quick check
display_cols = ["GEOID", "amenity_category"] + [f"{c}_std" for c in vars_to_std.keys()]
display(merged_df_std[display_cols].head())

,GEOID,amenity_category,gross_sq_ft_std,gini_index_std,white/non_hispanic_std,poverty_count_std,transit_commuters_std,population_total_std
0,06075012405,Public Services,2.188501,0.024567,-0.414069,0.337558,0.766497,-0.431016
1,06075012405,Infrastructure,0.659296,0.004455,-0.443895,0.152715,1.125936,-0.015195
2,06075012405,Public Services,3.912474,0.024567,-0.414069,0.337558,0.766497,-0.431016
3,06075012405,Public Services,2.476631,0.024567,-0.414069,0.337558,0.766497,-0.431016
4,06075012405,Public Services,0.174057,0.024567,-0.414069,0.337558,0.766497,-0.431016


In [101]:
merged_df_std["index_score"] = 0.3 * merged_df_std["gross_sq_ft_std"] + \
                               0.15 * merged_df_std["gini_index_std"] + \
                               0.15 * merged_df_std["white/non_hispanic_std"] + \
                               0.05 * merged_df_std["poverty_count_std"] + \
                               0.15 * merged_df_std["transit_commuters_std"] + \
                               0.2 * merged_df_std["population_total_std"]     

# display first few rows of index scores
display(merged_df_std[["GEOID", "amenity_category", "index_score"]].head())

# save to CSV for review
out_path = repo_root / "data" / "processed" / "amenity_category_geoid_index_detailed.csv"
merged_df_std.to_csv(out_path, index=False)
print(f"Saved category stats to {out_path}")

,GEOID,amenity_category,index_score
0,06075012405,Public Services,0.643774
1,06075012405,Infrastructure,0.305360
2,06075012405,Public Services,1.160966
3,06075012405,Public Services,0.730213
4,06075012405,Public Services,0.039441


Saved category stats to c:\Users\nehas\MTC-SF-Neighborhood-Amenity-Equity-Project\MTC-SF-Neighborhood-Amenity-Equity-Project\data\processed\amenity_category_geoid_index_detailed.csv


In [102]:
# compute mean index_score per GEOID within each amenity category
geoid_scores = (
    merged_df_std
    .dropna(subset=["index_score", "GEOID", "amenity_category"])
    .groupby(["amenity_category", "GEOID"], as_index=False)["index_score"]
    .mean()
    .rename(columns={"index_score": "mean_index_score"})
)

# helper to get GEOID closest to a target value
def _closest_geoid(df, target_col, value):
    df = df.copy()
    df["abs_diff"] = (df[target_col] - value).abs()
    row = df.loc[df["abs_diff"].idxmin()]
    return row["GEOID"], row[target_col]

rows = []
for cat, grp in geoid_scores.groupby("amenity_category", sort=True):
    n_tracts = grp["GEOID"].nunique()
    # max
    idx_max = grp["mean_index_score"].idxmax()
    geoid_max = grp.loc[idx_max, "GEOID"]
    max_score = grp.loc[idx_max, "mean_index_score"]
    # min
    idx_min = grp["mean_index_score"].idxmin()
    geoid_min = grp.loc[idx_min, "GEOID"]
    min_score = grp.loc[idx_min, "mean_index_score"]
    # median value and closest GEOID
    median_val = grp["mean_index_score"].median()
    geoid_med, med_score = _closest_geoid(grp, "mean_index_score", median_val)
    rows.append({
        "amenity_category": cat,
        "n_tracts": int(n_tracts),
        "geoid_max": str(geoid_max),
        "max_score": float(max_score),
        "geoid_min": str(geoid_min),
        "min_score": float(min_score),
        "geoid_median_closest": str(geoid_med),
        "median_score_value": float(med_score)
    })

category_stats_df = pd.DataFrame(rows).sort_values("amenity_category").reset_index(drop=True)

# display results and save to CSV for review
display(category_stats_df)
out_path = repo_root / "data" / "processed" / "amenity_category_geoid_index_stats.csv"
category_stats_df.to_csv(out_path, index=False)
print(f"Saved category stats to {out_path}")

,amenity_category,n_tracts,geoid_max,max_score,geoid_min,min_score,geoid_median_closest,median_score_value
0,Commercial,19,06075040200,1.046764,06075012406,-1.009047,06075025402,0.024361
1,Infrastructure,52,06075030700,1.018211,06075016101,-1.062253,06075023103,-0.130848
2,Public Services,172,06075016700,1.239798,06075061101,-1.420183,06075021200,-0.031399
3,Recreational,102,06075012405,1.905204,06075061101,-1.425284,06075032801,0.096343


Saved category stats to c:\Users\nehas\MTC-SF-Neighborhood-Amenity-Equity-Project\MTC-SF-Neighborhood-Amenity-Equity-Project\data\processed\amenity_category_geoid_index_stats.csv
